# Corso Python 2026 — Homework 2

# Esercizio 1

### Consegna: Gerarchia di utenti (classi e comportamenti)

Obiettivo:
Implementare una semplice gerarchia di classi che rappresentino utenti
di una piattaforma, con metodi per informazioni, permessi e creazione da
dizionari.

### Parte A — Classe base
- Definisci la classe `BaseUser` con:
	- attributo di classe `platform = "AIM_WebSite"`.
	- `__init__(self, username, email)` che salva `username`, `email`
		e imposta `self.active = True`.
	- `deactivate(self)` che imposta `self.active = False`.
	- `info(self)` che ritorna una stringa descrittiva non ambigua del tipo:
		`ClassName(username='...', email='...', platform='...', active=True)`.
	- `__str__(self)` che delega a `info()`.
	- `permissions(self)` che solleva `NotImplementedError` (da implementare
		nelle sottoclassi).

### Parte B — Sottoclassi concrete
- Implementa:
	- `User(BaseUser)`:
		- `permissions(self)` → `['read', 'comment']`.
		- `login_message(self)` → `f'{username} logged in as user'`.
	- `Admin(User)`:
		- estende i permessi di `User` con `['delete','ban','manage_users']`.
		- `login_message(self)` → `f'{username} logged in as admin'`.
	- `Guest(BaseUser)`:
		- `permissions(self)` → `['read']`.
		- override `info(self)` per aggiungere `guest=True` nella stringa.

---

### Spazio per la soluzione:


In [ ]:
class BaseUser:

	def __init__(self):
		pass

	def info(self):
		return (
			f"{self.__class__.__name__}("
			f"username={self.username!r}, "
			f"email={self.email!r}, "
			f"platform={self.platform!r}, "
			f"active={self.active})"
		)
	
	def __str__(self):
		return self.info()

	def permissions(self):
		raise NotImplementedError("Subclasses must implement permissions()")
	
class User():
	pass

class Admin():
	pass

class Guest():
	pass

In [ ]:
u = User("alice", "alice@example.com")
a = Admin("bob", "bob@example.com")
g = Guest("eve", "eve@example.com")

print(u.info())
print(a.info())
print(g.info())
print("---")
print(u.permissions())
print(a.permissions())
print(g.permissions())
print("---")
print(u.login_message())
print(a.login_message())
print("---")
u.deactivate()
print(u.info())

for user in [u, a, g]:
	print("---")
	print(user)
	print("permissions:", user.permissions())
	if hasattr(user, "login_message"):
		print(user.login_message())

User(username='alice', email='alice@example.com', platform='AIM_WebSite', active=True)
Admin(username='bob', email='bob@example.com', platform='AIM_WebSite', active=True)
Guest(username='eve', email='eve@example.com', platform='AIM_WebSite', active=True, guest=True)
---
['read', 'comment']
['read', 'comment', 'delete', 'ban', 'manage_users']
['read']
---
alice logged in as user
bob logged in as admin
---
User(username='alice', email='alice@example.com', platform='AIM_WebSite', active=False)
---
User(username='alice', email='alice@example.com', platform='AIM_WebSite', active=False)
permissions: ['read', 'comment']
alice logged in as user
---
Admin(username='bob', email='bob@example.com', platform='AIM_WebSite', active=True)
permissions: ['read', 'comment', 'delete', 'ban', 'manage_users']
bob logged in as admin
---
Guest(username='eve', email='eve@example.com', platform='AIM_WebSite', active=True, guest=True)
permissions: ['read']



## Esercizio 2

### Consegna:
Implementa un sistema di normalizzazione basato sul Pattern Strategy per dati numerici. Fornisci codice, semplici test e una breve spiegazione dei componenti.

### STEP 0 — Costruire le 3 strategie

- Implementa tre classi con lo stesso metodo `transform(self, data)` (accetta qualunque iterabile di numeri e ritorna una nuova lista):
	1) `IdentityNormalizer` — restituisce i dati invariati (nuova lista).
	2) `MinMaxNormalizer` — normalizza con `(x - min)/(max - min)`; se tutti i valori sono uguali o lista vuota, restituisce lista vuota o zeri coerenti con la lunghezza originale.
	3) `ZScoreNormalizer` — normalizza con `(x - mean)/std`; se `std == 0` o lista vuota, restituisce lista di zeri o lista vuota.

### STEP 1 — Costruire `DataProcessor`

- Implementa la classe `DataProcessor` con almeno:
	- `__init__(self, strategy)` che salva l'oggetto strategia (qualsiasi oggetto che implementi `transform`).
	- `process(self, data)` che delega alla strategia (`return self.strategy.transform(data)`).
	- `set_strategy(self, strategy)` per sostituire la strategia a runtime (accetta oggetto strategia o, opzionalmente, valore risolvibile dalla factory).

### STEP 2 — Costruire `Normalizer` con `create` da dizionario

- Implementa una classe `Normalizer` che mantenga un dizionario interno (registry) che mappa nomi stringa a classi strategia.
- Fornisci `@classmethod def create(cls, name):` che instanzia e ritorna la strategia corrispondente guardando il registry (es. `Normalizer.create('minmax')`).
- Il registry è un semplice dizionario: `cls._registry = { 'identity': IdentityNormalizer, ... }`.

### STEP 3 — Costruire la Factory con `from_name` in `DataProcessor`

- Aggiungi a `DataProcessor` un metodo di classe `from_name(cls, name)` che costruisca un `DataProcessor` risolvendo la strategia tramite `Normalizer.create(name)` (es. `DataProcessor.from_name('zscore')`).
- Questo separa la logica di risoluzione dal codice client e rende l'uso più comodo in fase di test e demo.

### STEP 4 — (Facoltativo) Decoratore `register`

- (Opzionale) Implementa un decoratore `@Normalizer.register(name)` che registra automaticamente una classe strategia nel `Normalizer._registry` durante la sua definizione. Questo rende l'estensione del sistema semplice e leggibile.

---

### Spazio di lavoro:


In [ ]:
class Normalizer:
	_registry = {}

	@classmethod
	def register(cls, name):
		"""
		Decoratore di classe:
		registra automaticamente una strategia nel registry.
		"""
		raise NotImplementedError("STEP 4 IMPLEMENTATION")

	@classmethod
	def create(cls, name):
		"""
		Factory method: crea una strategia a partire dal nome.
		"""
		raise NotImplementedError("STEP 2 IMPLEMENTATION")

	def transform(self, data):
		"""
		Metodo da ridefinire nelle classi derivate.
		"""
		raise NotImplementedError("Subclasses must implement transform")

class DataProcessor:
	pass

# ------------------------------------------------------------

import math

class IdentityNormalizer(Normalizer):
	pass

class MinMaxNormalizer(Normalizer):
	pass

class ZScoreNormalizer(Normalizer):
	pass


In [28]:
data = [10, 20, 30, 40]

print("STEP 1: verifiche minime")

processor = DataProcessor(IdentityNormalizer())
print("Identity:", processor.process(data))

processor = DataProcessor(MinMaxNormalizer())
print("MinMax:", processor.process(data))

processor = DataProcessor(ZScoreNormalizer())
print("ZScore:", processor.process(data))


print("\nSTEP 2: verifiche minime")

processor = DataProcessor(Normalizer.create("identity"))
print("Identity:", processor.process(data))

processor = DataProcessor(Normalizer.create("minmax"))
print("MinMax:", processor.process(data))

processor = DataProcessor(Normalizer.create("zscore"))
print("ZScore:", processor.process(data))


print("\nSTEP 3: verifiche minime")

processor = DataProcessor.from_name("identity")
print("Identity:", processor.process(data))

processor.set_strategy(Normalizer.create("minmax"))
print("MinMax:", processor.process(data))

processor.set_strategy(Normalizer.create("zscore"))
print("ZScore:", processor.process(data))

STEP 1: verifiche minime


TypeError: DataProcessor() takes no arguments

## Esercizio 3

### Consegna:
Implementa una classe `Complex` didattica per esercitarti con i dunder methods e con il comportamento degli operatori in Python.

Nota: non usare il tipo built-in `complex` internamente per fare i conti (puoi usarlo solo nei test, per confronti).

### Parte A — Struttura base e rappresentazione
1. Definisci una classe `Complex` con attributi di istanza:
	- `re` (parte reale)
	- `im` (parte immaginaria)
2. Implementa `__init__(self, re=0.0, im=0.0)`:
	- accetta `int` o `float`
	- salva sempre come `float`
3. Implementa `__repr__` in forma non ambigua:
	- `Complex(re=..., im=...)`
4. Implementa `__str__` in forma matematica:
	- `a+bi` oppure `a-bi`
	- casi semplici da gestire:
	  - `im == 0` → `"a"`
	  - `re == 0` → `"bi"` (con segno corretto)
	  - `re == 0` e `im == 0` → `"0"`

### Parte B — Uguaglianza e hashing
5. Implementa `__eq__`:
	- `Complex == Complex` confrontando `re` e `im`
	- `Complex == numero reale` (`int`/`float`) interpretato come `x + 0i`
	- altrimenti ritorna `NotImplemented`
6. Implementa `__hash__` coerente con `__eq__`.

### Parte C — Operazioni aritmetiche
7. Implementa `__add__` e `__radd__`:
	- `Complex + Complex`
	- `Complex + reale`
	- `reale + Complex`
8. Implementa `__sub__` e `__rsub__` in modo analogo.
9. Implementa `__mul__` e `__rmul__`:
	- $(a+bi)(c+di) = (ac-bd) + (ad+bc)i$
	- supporta anche moltiplicazione per reale
10. Implementa `__truediv__` e `__rtruediv__`:
	- $(a+bi)/(c+di) = \frac{(ac+bd) + (bc-ad)i}{c^2+d^2}$
	- se il denominatore è `0+0i`, solleva `ZeroDivisionError`
	- supporta divisione per reale e reale diviso `Complex`
11. Implementa `__neg__`:
	- `-(a+bi) = (-a) + (-b)i`

### Parte D — Metodi comodi e conversioni
12. Implementa `conjugate(self)` che ritorna il coniugato.
13. Implementa `__abs__`:
	- $|a+bi| = \sqrt{a^2+b^2}$ (puoi usare `math.sqrt`)

Vincoli:
- Non usare `itertools` o moduli esterni.
- Non usare internamente il tipo `complex` per implementare le operazioni.
- Implementa i dunder richiesti per ottenere il comportamento corretto degli operatori.

---

### Spazio di lavoro:


In [ ]:
import math # usa solo per sqrt

class Complex:
	def __init__(self, re=0.0, im=0.0):
	   pass

	def __complex__(self): # Questa serve solo per i test finali. NON USARE
		return complex(self.re, self.im)


In [ ]:
z = Complex(2, 3)
w = Complex(1, -4)

print("repr(z):", repr(z))
print("str(z):", str(z))

print("")

print("z + w =", z + w)
print("z - w =", z - w)
print("z * w =", z * w)
print("z / w =", z / w)

print("")

print("2 + z =", 2 + z)
print("2 - z =", 2 - z)
print("2 * z =", 2 * z)
print("2 / z =", 2 / z)

print("")

print("z == Complex(2,3):", z == Complex(2, 3))
print("z == 2:", z == 2)
print("Complex(2,0) == 2:", Complex(2, 0) == 2)

print("")

print("abs(Complex(3,4)):", abs(Complex(3, 4)))
print("conjugate:", z.conjugate())

repr(z): <__main__.Complex object at 0x70cfd7a9c290>
str(z): <__main__.Complex object at 0x70cfd7a9c290>



TypeError: unsupported operand type(s) for +: 'Complex' and 'Complex'